In [ ]:
!pip install -q transformers datasets evaluate seqeval accelerate torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00


In [ ]:

import datasets
import transformers


In [ ]:

from tqdm import tqdm
for i in tqdm(range(100),disable=True):
    pass

In [ ]:
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)

In [ ]:
dataset=load_dataset("lhoestq/conll2003")
dataset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/281k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [ ]:
print(dataset["train"].features)

{'id': Value('string'), 'tokens': List(Value('string')), 'pos_tags': List(Value('int64')), 'chunk_tags': List(Value('int64')), 'ner_tags': List(Value('int64'))}


In [ ]:
chunk_labels = [
    "O", "B-ADJP", "I-ADJP", "B-ADVP", "I-ADVP", "B-CONJP", "I-CONJP",
    "B-INTJ", "I-INTJ", "B-LST", "I-LST", "B-NP", "I-NP", "B-PP", "I-PP",
    "B-PRT", "I-PRT", "B-SBAR", "I-SBAR", "B-UCP", "I-UCP", "B-VP", "I-VP"
]

pos_labels = [
    "``", "''", "#", "$", "''", "(", ")", ",", ".", ":", "CC", "CD",
    "DT", "EX", "FW", "IN", "JJ", "JJR", "JJS", "LS", "MD", "NN", "NNS",
    "NNP", "NNPS", "PDT", "POS", "PRP", "PRPS", "RB", "RBR", "RBS", "RP",
    "SYM", "TO", "UH", "VB", "VBD", "VBG", "VBN", "VBP", "VBZ", "WDT",
    "WP", "WP$", "WRB"
]
print(len(pos_labels))
print(len(chunk_labels))

46
23


In [ ]:
id2label = {i: label for i, label in enumerate(chunk_labels)}
label2id = {label: i for i, label in enumerate(chunk_labels)}
print("id2label:", id2label)
print("label2id:", label2id)

id2label: {0: 'O', 1: 'B-ADJP', 2: 'I-ADJP', 3: 'B-ADVP', 4: 'I-ADVP', 5: 'B-CONJP', 6: 'I-CONJP', 7: 'B-INTJ', 8: 'I-INTJ', 9: 'B-LST', 10: 'I-LST', 11: 'B-NP', 12: 'I-NP', 13: 'B-PP', 14: 'I-PP', 15: 'B-PRT', 16: 'I-PRT', 17: 'B-SBAR', 18: 'I-SBAR', 19: 'B-UCP', 20: 'I-UCP', 21: 'B-VP', 22: 'I-VP'}
label2id: {'O': 0, 'B-ADJP': 1, 'I-ADJP': 2, 'B-ADVP': 3, 'I-ADVP': 4, 'B-CONJP': 5, 'I-CONJP': 6, 'B-INTJ': 7, 'I-INTJ': 8, 'B-LST': 9, 'I-LST': 10, 'B-NP': 11, 'I-NP': 12, 'B-PP': 13, 'I-PP': 14, 'B-PRT': 15, 'I-PRT': 16, 'B-SBAR': 17, 'I-SBAR': 18, 'B-UCP': 19, 'I-UCP': 20, 'B-VP': 21, 'I-VP': 22}


In [ ]:

model_checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(chunk_labels),
    id2label=id2label,
    label2id=label2id,
)

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
print("Model checkpoint:", model_checkpoint)
print("num_labels:", model.config.num_labels)
print("id2label:", model.config.id2label)
print("label2id:", model.config.label2id)

Model checkpoint: distilbert-base-cased
num_labels: 23
id2label: {0: 'O', 1: 'B-ADJP', 2: 'I-ADJP', 3: 'B-ADVP', 4: 'I-ADVP', 5: 'B-CONJP', 6: 'I-CONJP', 7: 'B-INTJ', 8: 'I-INTJ', 9: 'B-LST', 10: 'I-LST', 11: 'B-NP', 12: 'I-NP', 13: 'B-PP', 14: 'I-PP', 15: 'B-PRT', 16: 'I-PRT', 17: 'B-SBAR', 18: 'I-SBAR', 19: 'B-UCP', 20: 'I-UCP', 21: 'B-VP', 22: 'I-VP'}
label2id: {'O': 0, 'B-ADJP': 1, 'I-ADJP': 2, 'B-ADVP': 3, 'I-ADVP': 4, 'B-CONJP': 5, 'I-CONJP': 6, 'B-INTJ': 7, 'I-INTJ': 8, 'B-LST': 9, 'I-LST': 10, 'B-NP': 11, 'I-NP': 12, 'B-PP': 13, 'I-PP': 14, 'B-PRT': 15, 'I-PRT': 16, 'B-SBAR': 17, 'I-SBAR': 18, 'B-UCP': 19, 'I-UCP': 20, 'B-VP': 21, 'I-VP': 22}


In [ ]:
sample_tokens = dataset["train"][0]["tokens"]
sample_chunk_tags = dataset["train"][0]["chunk_tags"]
print("Tokens:", sample_tokens)
print("Chunk Tag IDs:", sample_chunk_tags)
print("Chunk Tag Names:", [chunk_labels[i] for i in sample_chunk_tags])

Tokens: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Chunk Tag IDs: [11, 21, 11, 12, 21, 22, 11, 12, 0]
Chunk Tag Names: ['B-NP', 'B-VP', 'B-NP', 'I-NP', 'B-VP', 'I-VP', 'B-NP', 'I-NP', 'O']


In [ ]:
encoded_sample = tokenizer(sample_tokens, is_split_into_words=True, truncation=True)
print("Input IDs:", encoded_sample["input_ids"])
print("Attention Mask:", encoded_sample["attention_mask"])
print("Tokens:", tokenizer.convert_ids_to_tokens(encoded_sample["input_ids"]))
print("Word IDs:", encoded_sample.word_ids())

Input IDs: [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Tokens: ['[CLS]', 'EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'la', '##mb', '.', '[SEP]']
Word IDs: [None, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, None]


In [ ]:

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
    )
    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs


In [ ]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
tokenized_datasets

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

In [ ]:
print("input_ids:", tokenized_datasets["train"][0]["input_ids"][:30])
print("attention_mask:", tokenized_datasets["train"][0]["attention_mask"][:30])
print("labels:", tokenized_datasets["train"][0]["labels"][:30])

input_ids: [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
labels: [-100, 11, 21, 11, 12, 21, 22, 11, 12, -100, 0, -100]


In [ ]:
tokens = tokenizer.convert_ids_to_tokens(tokenized_datasets["train"][0]["input_ids"])
labels = tokenized_datasets["train"][0]["labels"]
for tok, lab in zip(tokens[:40], labels[:40]):
    if lab == -100:
        print(f"{tok:15} -> -100")
    else:
        print(f"{tok:15} -> {id2label[lab]}")

[CLS]           -> -100
EU              -> B-NP
rejects         -> B-VP
German          -> B-NP
call            -> I-NP
to              -> B-VP
boycott         -> I-VP
British         -> B-NP
la              -> I-NP
##mb            -> -100
.               -> O
[SEP]           -> -100


In [ ]:
seqeval = evaluate.load("seqeval")

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [chunk_labels[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [chunk_labels[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:

training_args = TrainingArguments(
    output_dir="./distilbert-chunking",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.324491,0.182275,0.913871,0.905169,0.909499,0.952633
2,0.147621,0.163801,0.919079,0.911970,0.915511,0.955614
3,0.116182,0.161368,0.919525,0.915702,0.917609,0.956998


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2634, training_loss=0.19609825812059545, metrics={'train_runtime': 263.3131, 'train_samples_per_second': 159.973, 'train_steps_per_second': 10.003, 'total_flos': 524901061281240.0, 'train_loss': 0.19609825812059545, 'epoch': 3.0})

In [ ]:
validation_results = trainer.evaluate()
validation_results

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 0.16136802732944489,
 'eval_precision': 0.9195254263747414,
 'eval_recall': 0.9157015157403808,
 'eval_f1': 0.9176094872743559,
 'eval_accuracy': 0.9569978372269743,
 'eval_runtime': 10.125,
 'eval_samples_per_second': 320.987,
 'eval_steps_per_second': 20.148,
 'epoch': 3.0}

In [ ]:
test_results = trainer.evaluate(tokenized_datasets["test"])
test_results


{'eval_loss': 0.18764127790927887,
 'eval_precision': 0.9095035070829321,
 'eval_recall': 0.9011173691860465,
 'eval_f1': 0.9052910173629332,
 'eval_accuracy': 0.9534653252041277,
 'eval_runtime': 8.272,
 'eval_samples_per_second': 417.433,
 'eval_steps_per_second': 26.112,
 'epoch': 3.0}

In [ ]:
print("Precision:", round(test_results["eval_precision"], 4))
print("Recall:", round(test_results["eval_recall"], 4))
print("F1 Score:", round(test_results["eval_f1"], 4))
print("Accuracy:", round(test_results["eval_accuracy"], 4))

Precision: 0.9095
Recall: 0.9011
F1 Score: 0.9053
Accuracy: 0.9535


In [ ]:
trainer.save_model("./distilbert-chunking-final")
tokenizer.save_pretrained("./distilbert-chunking-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./distilbert-chunking-final/tokenizer_config.json',
 './distilbert-chunking-final/tokenizer.json')

In [ ]:
chunking_pipeline = pipeline(
    "token-classification",
    model="./distilbert-chunking-final",
    tokenizer="./distilbert-chunking-final",
    aggregation_strategy="simple",
)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [ ]:
text = "The quick brown fox jumps over the lazy dog"
predictions = chunking_pipeline(text)
print("Input:", text)
for entity in predictions:
    print(f"Word/Phrase: {entity['word']} | Tag: {entity['entity_group']} | Score: {entity['score']:.4f}")

Input: The quick brown fox jumps over the lazy dog
Word/Phrase: The quick brown fox | Tag: NP | Score: 0.9959
Word/Phrase: jumps | Tag: VP | Score: 0.9549
Word/Phrase: over | Tag: PP | Score: 0.8663
Word/Phrase: the lazy dog | Tag: NP | Score: 0.9978


In [ ]:
pos_id2label = {i: label for i, label in enumerate(pos_labels)}
pos_label2id = {label: i for i, label in enumerate(pos_labels)}
print("POS id2label:", pos_id2label)


POS id2label: {0: '``', 1: "''", 2: '#', 3: '$', 4: "''", 5: '(', 6: ')', 7: ',', 8: '.', 9: ':', 10: 'CC', 11: 'CD', 12: 'DT', 13: 'EX', 14: 'FW', 15: 'IN', 16: 'JJ', 17: 'JJR', 18: 'JJS', 19: 'LS', 20: 'MD', 21: 'NN', 22: 'NNS', 23: 'NNP', 24: 'NNPS', 25: 'PDT', 26: 'POS', 27: 'PRP', 28: 'PRPS', 29: 'RB', 30: 'RBR', 31: 'RBS', 32: 'RP', 33: 'SYM', 34: 'TO', 35: 'UH', 36: 'VB', 37: 'VBD', 38: 'VBG', 39: 'VBN', 40: 'VBP', 41: 'VBZ', 42: 'WDT', 43: 'WP', 44: 'WP$', 45: 'WRB'}


In [ ]:
sample_pos_tags = dataset["train"][0]["pos_tags"]
print("Tokens:", sample_tokens)
print("POS Tag IDs:", sample_pos_tags)
print("POS Tag Names:", [pos_labels[i] for i in sample_pos_tags])

Tokens: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
POS Tag IDs: [22, 42, 16, 21, 35, 37, 16, 21, 7]
POS Tag Names: ['NNS', 'WDT', 'JJ', 'NN', 'UH', 'VBD', 'JJ', 'NN', ',']


In [ ]:
for token, pos_tag, chunk_tag in zip(
    dataset["train"][0]["tokens"],
    dataset["train"][0]["pos_tags"],
    dataset["train"][0]["chunk_tags"]
):
    print(
        f"Token: {token:15} | POS: {pos_labels[pos_tag]:10} | Chunk: {chunk_labels[chunk_tag]}"
    )

Token: EU              | POS: NNS        | Chunk: B-NP
Token: rejects         | POS: WDT        | Chunk: B-VP
Token: German          | POS: JJ         | Chunk: B-NP
Token: call            | POS: NN         | Chunk: I-NP
Token: to              | POS: UH         | Chunk: B-VP
Token: boycott         | POS: VBD        | Chunk: I-VP
Token: British         | POS: JJ         | Chunk: B-NP
Token: lamb            | POS: NN         | Chunk: I-NP
Token: .               | POS: ,          | Chunk: O


POS Tagging vs Chunking

POS Tagging:
Assigns grammatical labels to each word (e.g., John → Noun, works → Verb). It is easier as it focuses on individual words with limited context.

Chunking:
Groups words into meaningful phrases (e.g., “The quick brown fox” → Noun Phrase). It is more complex as it identifies phrase boundaries using BIO tags.

Challenges

Models like BERT use subword tokenization (e.g., tokenization → token, ##iza, ##tion), causing label mismatch.
Solution: assign the label to the first token and use -100 for remaining tokens.

Observations
DistilBERT gives faster training with good accuracy
Evaluated using seqeval (Precision, Recall, F1 Score)
POS tagging is easier, while chunking is harder due to phrase-level understanding

Insights

The model correctly identifies basic word categories (nouns, adjectives, verbs)and forms meaningful chunks like noun phrases (NP) and verb phrases (VP).

Phrase grouping is mostly accurate (e.g., “German call”, “British lamb” as noun phrases).

Some POS tags are misclassified (e.g., “rejects”, “to”), showing limitations in contextual understanding.

Chunk predictions are relatively better than POS in some cases, indicating the model captures phrase structure patterns effectively.

Errors highlight the need for better fine-tuning or more training data for improved accuracy.